# 🏃 MHEALTH Dataset — Multi-Model Comparison
### CNN · BiGRU · CNN+BiGRU · CNN+BiLSTM · CNN+BiGRU+Gated Attention
**Human Activity Recognition | No Data Leakage | Subject-Level Splits**

**Dataset:** MHEALTH — 10 subjects · 12 activities · 23 sensor channels · 50 Hz

**Sensors:** Chest accelerometer, Left-ankle (acc + gyro + mag), Right-arm (acc + gyro + mag), ECG (2 leads)

**Models compared:**
1. **CNN Only** — 1D Convolutional feature extractor
2. **BiGRU Only** — Bidirectional GRU temporal model
3. **CNN + BiGRU** — Local features → temporal modelling (no attention)
4. **CNN + BiLSTM** — Local features → LSTM temporal modelling (no attention)
5. **CNN + BiGRU + Gated Attention** ← Full hybrid (flagship model)

**No-Leakage guarantee:** Train / Val / Test splits at the **subject level** — no subject appears in more than one split.


## 📦 Step 1 — Install & Import Libraries

In [ ]:
!pip install -q scikit-learn matplotlib seaborn tensorflow

import os, zipfile, random, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter, defaultdict

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, roc_curve, auc,
    cohen_kappa_score, matthews_corrcoef,
    precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


## 📂 Step 2 — Upload & Extract Dataset
Upload your `mhealth_dataset.zip` when prompted.


In [ ]:
from google.colab import files

print('Please upload mhealth_dataset.zip ...')
uploaded = files.upload()

zip_name    = list(uploaded.keys())[0]
extract_dir = '/content/MHEALTH'

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_dir)

# Locate the directory containing the .log files
def find_mhealth_root(base):
    for root, dirs, flist in os.walk(base):
        logs = [f for f in flist if f.startswith('mHealth_subject') and f.endswith('.log')]
        if logs:
            return root
    raise FileNotFoundError('mHealth_subject*.log not found — check zip structure.')

DATASET_PATH = find_mhealth_root(extract_dir)
log_files = sorted([f for f in os.listdir(DATASET_PATH) if f.startswith('mHealth_subject')])
print(f'Dataset root: {DATASET_PATH}')
print(f'Subject files found ({len(log_files)}): {log_files}')


## 🔬 Step 3 — Dataset Constants & Column Definitions

**23 sensor columns** (column indices 0–22) + label (index 23):

| Columns | Sensor |
|---------|--------|
| 0–2     | Chest acceleration (X, Y, Z) |
| 3–4     | ECG lead 1 & 2 |
| 5–7     | Left-ankle acceleration (X, Y, Z) |
| 8–10    | Left-ankle gyroscope (X, Y, Z) |
| 11–13   | Left-ankle magnetometer (X, Y, Z) |
| 14–16   | Right-lower-arm acceleration (X, Y, Z) |
| 17–19   | Right-lower-arm gyroscope (X, Y, Z) |
| 20–22   | Right-lower-arm magnetometer (X, Y, Z) |
| **23**  | **Activity label** |

**Activities (null class 0 excluded):**
1 Standing still · 2 Sitting · 3 Lying down · 4 Walking · 5 Climbing stairs ·
6 Waist bends forward · 7 Frontal elevation of arms · 8 Knees bending ·
9 Cycling · 10 Jogging · 11 Running · 12 Jump front & back


In [ ]:
# ── Dataset configuration ───────────────────────────────────────────────────
N_SUBJECTS   = 10
N_CLASSES    = 12            # exclude null class (label 0)
N_CHANNELS   = 23            # all sensor channels
SAMPLING_HZ  = 50
WINDOW_SIZE  = 128           # 2.56 seconds @ 50 Hz  (same as UCI HAR)
STEP_SIZE    = 64            # 50 % overlap

ACTIVITY_LABELS = {
    1: 'Standing',
    2: 'Sitting',
    3: 'Lying down',
    4: 'Walking',
    5: 'Climbing stairs',
    6: 'Waist bends fwd',
    7: 'Arms elevation',
    8: 'Knees bending',
    9: 'Cycling',
    10: 'Jogging',
    11: 'Running',
    12: 'Jump fwd/back',
}

CHANNEL_NAMES = [
    'chest_acc_x', 'chest_acc_y', 'chest_acc_z',
    'ecg_lead1', 'ecg_lead2',
    'ankle_acc_x', 'ankle_acc_y', 'ankle_acc_z',
    'ankle_gyro_x', 'ankle_gyro_y', 'ankle_gyro_z',
    'ankle_mag_x', 'ankle_mag_y', 'ankle_mag_z',
    'arm_acc_x', 'arm_acc_y', 'arm_acc_z',
    'arm_gyro_x', 'arm_gyro_y', 'arm_gyro_z',
    'arm_mag_x', 'arm_mag_y', 'arm_mag_z',
]

print(f'Window size : {WINDOW_SIZE} samples = {WINDOW_SIZE/SAMPLING_HZ:.2f} s')
print(f'Step size   : {STEP_SIZE} samples  (50 % overlap)')
print(f'N channels  : {N_CHANNELS}')
print(f'N classes   : {N_CLASSES} (null class excluded)')


## 📥 Step 4 — Load Subject Files & Sliding-Window Segmentation

In [ ]:
def load_subject(filepath):
    """Load a single subject .log file.  Returns (data: ndarray (N,23), labels: ndarray (N,))."""
    df = pd.read_csv(filepath, sep='\t', header=None)
    # Last column is the label; first 23 are sensor channels
    data   = df.iloc[:, :N_CHANNELS].values.astype(np.float32)
    labels = df.iloc[:,  N_CHANNELS].values.astype(int)
    return data, labels


def sliding_window(data, labels, window=WINDOW_SIZE, step=STEP_SIZE):
    """
    Segment a continuous time series into overlapping windows.
    Window label = majority vote of sample labels within the window.
    Windows containing the null class (0) are discarded.
    """
    X_wins, y_wins = [], []
    n = len(data)
    for start in range(0, n - window + 1, step):
        end     = start + window
        seg     = data[start:end]              # (window, n_channels)
        seg_lab = labels[start:end]
        # Majority vote
        counts  = Counter(seg_lab)
        maj_lab = counts.most_common(1)[0][0]
        if maj_lab == 0:
            continue                            # discard null class
        X_wins.append(seg)
        y_wins.append(maj_lab)
    return np.array(X_wins, dtype=np.float32), np.array(y_wins, dtype=int)


# ── Load all subjects ──────────────────────────────────────────────────────────
all_X, all_y, all_subjects = [], [], []

for subj_id in range(1, N_SUBJECTS + 1):
    fname = f'mHealth_subject{subj_id}.log'
    fpath = os.path.join(DATASET_PATH, fname)
    raw_data, raw_labels = load_subject(fpath)

    X_wins, y_wins = sliding_window(raw_data, raw_labels)
    all_X.append(X_wins)
    all_y.append(y_wins)
    all_subjects.append(np.full(len(y_wins), subj_id, dtype=int))

    print(f'  Subject {subj_id:2d}: raw={len(raw_data):7,} samples → {len(X_wins):4d} windows')

all_X        = np.concatenate(all_X,        axis=0)   # (total_windows, 128, 23)
all_y        = np.concatenate(all_y,        axis=0)   # (total_windows,)
all_subjects = np.concatenate(all_subjects, axis=0)   # (total_windows,)

# Convert to 0-indexed labels (1–12 → 0–11)
all_y = all_y - 1

print(f'\nTotal windows  : {len(all_X):,}')
print(f'Data shape     : {all_X.shape}')
print(f'Label shape    : {all_y.shape}')
print(f'Classes present: {sorted(np.unique(all_y))}')
print(f'Subject ids    : {sorted(np.unique(all_subjects))}')


## 🛡️ Step 5 — Subject-Level Split (No Data Leakage)

10 subjects → 7 train · 1 val · 2 test


In [ ]:
np.random.seed(SEED)
all_subj_ids = sorted(np.unique(all_subjects))          # [1..10]

# Fixed split for reproducibility: subjects 9 & 10 → test, subject 8 → val
test_subjects = [9, 10]
val_subjects  = [8]
tr_subjects   = [s for s in all_subj_ids if s not in test_subjects + val_subjects]

print(f'Train subjects ({len(tr_subjects)}): {tr_subjects}')
print(f'Val   subjects ({len(val_subjects)}): {val_subjects}')
print(f'Test  subjects ({len(test_subjects)}): {test_subjects}')

# Verify no overlap
assert not (set(tr_subjects) & set(val_subjects) & set(test_subjects)), 'DATA LEAK!'
print('✅ No subject overlap across splits.')

tr_mask  = np.isin(all_subjects, tr_subjects)
val_mask = np.isin(all_subjects, val_subjects)
te_mask  = np.isin(all_subjects, test_subjects)

X_train, y_train = all_X[tr_mask],  all_y[tr_mask]
X_val,   y_val   = all_X[val_mask], all_y[val_mask]
X_test,  y_test  = all_X[te_mask],  all_y[te_mask]

print(f'\nFinal splits:')
print(f'  Train : {X_train.shape}  →  {len(X_train):,} windows')
print(f'  Val   : {X_val.shape}  →  {len(X_val):,} windows')
print(f'  Test  : {X_test.shape}  →  {len(X_test):,} windows')


## 📊 Step 6 — Per-Channel Z-Score Normalization (Fit on Train Only)

In [ ]:
train_mean = X_train.mean(axis=(0, 1), keepdims=True)   # (1, 1, 23)
train_std  = X_train.std( axis=(0, 1), keepdims=True) + 1e-8

X_train = (X_train - train_mean) / train_std
X_val   = (X_val   - train_mean) / train_std
X_test  = (X_test  - train_mean) / train_std

y_train_cat = to_categorical(y_train, N_CLASSES)
y_val_cat   = to_categorical(y_val,   N_CLASSES)
y_test_cat  = to_categorical(y_test,  N_CLASSES)

print('Normalization done (fit on train only — no leakage).')
print(f'X_train range : [{X_train.min():.3f}, {X_train.max():.3f}]')
print(f'X_val   range : [{X_val.min():.3f},   {X_val.max():.3f}]')
print(f'X_test  range : [{X_test.min():.3f},  {X_test.max():.3f}]')


## 🔍 Step 7 — Exploratory Data Analysis

In [ ]:
activity_names = [ACTIVITY_LABELS[i+1] for i in range(N_CLASSES)]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
colors = plt.cm.tab20(np.linspace(0, 1, N_CLASSES))

for ax, (y_split, title) in zip(axes, [
    (y_train, 'Train'), (y_val, 'Validation'), (y_test, 'Test')
]):
    counts = [np.sum(y_split == i) for i in range(N_CLASSES)]
    bars = ax.bar(activity_names, counts, color=colors)
    ax.set_title(f'{title} Set — Class Distribution', fontsize=13, fontweight='bold')
    ax.set_xticklabels(activity_names, rotation=40, ha='right', fontsize=8)
    ax.set_ylabel('Count')
    for bar, c in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(c), ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('mhealth_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Signal sample plot ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(5, 5, figsize=(20, 12))
sample_idx = 0
for i, ax in enumerate(axes.flatten()):
    if i >= N_CHANNELS:
        ax.axis('off'); continue
    ax.plot(X_train[sample_idx, :, i], linewidth=1.0, color=f'C{i % 10}')
    ax.set_title(CHANNEL_NAMES[i], fontsize=7, fontweight='bold')
    ax.set_xlabel('Timestep', fontsize=6)
    ax.tick_params(labelsize=6)
    ax.grid(alpha=0.3)

activity = ACTIVITY_LABELS[y_train[sample_idx] + 1]
fig.suptitle(f'All 23 Sensor Channels — Sample Activity: {activity}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('mhealth_signal_sample.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Windows per activity ───────────────────────────────────────────────────────
print('\nWindows per activity (train):')
for i in range(N_CLASSES):
    n = np.sum(y_train == i)
    print(f'  {ACTIVITY_LABELS[i+1]:<22} : {n:4d}')


## 🏗️ Step 8 — Shared Components

### Gated Attention Mechanism
```
H: (batch, T, d)
  score  = tanh(H @ W_h)           ← content relevance
  gate   = sigmoid(H @ W_g)        ← information gate
  alpha  = softmax(score ⊙ gate)   ← gated attention weights
  context= Σ_t(alpha_t × H_t)      ← weighted context vector
```


In [ ]:
class GatedAttention(layers.Layer):
    """
    Gated Attention Mechanism.
    Input : H of shape (batch, T, d)
    Output: context (batch, d), attention weights (batch, T)
    """
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        d = input_shape[-1]
        self.W_h = self.add_weight(shape=(d, self.units), name='W_h',
                                   initializer='glorot_uniform', trainable=True)
        self.W_g = self.add_weight(shape=(d, self.units), name='W_g',
                                   initializer='glorot_uniform', trainable=True)
        self.v   = self.add_weight(shape=(self.units, 1), name='v',
                                   initializer='glorot_uniform', trainable=True)
        super().build(input_shape)

    def call(self, H):
        score   = tf.tanh(tf.matmul(H, self.W_h))      # (batch, T, units)
        gate    = tf.sigmoid(tf.matmul(H, self.W_g))   # (batch, T, units)
        gated   = score * gate
        e       = tf.matmul(gated, self.v)              # (batch, T, 1)
        attn_w  = tf.nn.softmax(e, axis=1)              # (batch, T, 1)
        context = tf.reduce_sum(attn_w * H, axis=1)    # (batch, d)
        return context, tf.squeeze(attn_w, axis=-1)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'units': self.units})
        return cfg


def cnn_block(x, filters_1, filters_2, pool_size=2, dropout=0.2, block_id=1):
    """Shared CNN block: 2× Conv1D → BN → ReLU → MaxPool → Dropout."""
    prefix = f'block{block_id}'
    x = layers.Conv1D(filters_1, 3, padding='same', name=f'{prefix}_conv1')(x)
    x = layers.BatchNormalization(name=f'{prefix}_bn1')(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv1D(filters_2, 3, padding='same', name=f'{prefix}_conv2')(x)
    x = layers.BatchNormalization(name=f'{prefix}_bn2')(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(pool_size=pool_size, name=f'{prefix}_pool')(x)
    x = layers.Dropout(dropout)(x)
    return x


print('GatedAttention layer & helper functions defined.')


## 🔨 Step 9 — Model Builders (5 Architectures)

In [ ]:
INPUT_SHAPE = (WINDOW_SIZE, N_CHANNELS)   # (128, 23)
DR = 0.4                                  # global dropout rate


# ══════════════════════════════════════════════════════════════════════════════
# 1. CNN Only
# ══════════════════════════════════════════════════════════════════════════════
def build_cnn_only(input_shape=INPUT_SHAPE, n_classes=N_CLASSES, dr=DR):
    inp = layers.Input(shape=input_shape, name='input')
    x   = cnn_block(inp, 64,  64,  block_id=1)
    x   = cnn_block(x,  128, 128,  block_id=2)
    x   = cnn_block(x,  256, 256,  block_id=3)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dropout(dr)(x)
    x   = layers.Dense(128, name='fc1')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation('relu')(x)
    x   = layers.Dropout(dr / 2)(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)
    return Model(inputs=inp, outputs=out, name='CNN_Only')


# ══════════════════════════════════════════════════════════════════════════════
# 2. BiGRU Only
# ══════════════════════════════════════════════════════════════════════════════
def build_bigru_only(input_shape=INPUT_SHAPE, n_classes=N_CLASSES, dr=DR):
    inp = layers.Input(shape=input_shape, name='input')
    x   = layers.Bidirectional(
              layers.GRU(128, return_sequences=True, dropout=0.2), name='bigru_1')(inp)
    x   = layers.Bidirectional(
              layers.GRU(64,  return_sequences=False, dropout=0.2), name='bigru_2')(x)
    x   = layers.Dropout(dr)(x)
    x   = layers.Dense(128, name='fc1')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation('relu')(x)
    x   = layers.Dropout(dr / 2)(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)
    return Model(inputs=inp, outputs=out, name='BiGRU_Only')


# ══════════════════════════════════════════════════════════════════════════════
# 3. CNN + BiGRU  (no attention)
# ══════════════════════════════════════════════════════════════════════════════
def build_cnn_bigru(input_shape=INPUT_SHAPE, n_classes=N_CLASSES, dr=DR):
    inp = layers.Input(shape=input_shape, name='input')
    x   = cnn_block(inp, 64,  64,  block_id=1)
    x   = cnn_block(x,  128, 128,  block_id=2)
    x   = layers.Bidirectional(
              layers.GRU(128, return_sequences=True, dropout=0.2), name='bigru_1')(x)
    x   = layers.Bidirectional(
              layers.GRU(64,  return_sequences=False, dropout=0.2), name='bigru_2')(x)
    x   = layers.Dropout(dr)(x)
    x   = layers.Dense(128, name='fc1')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation('relu')(x)
    x   = layers.Dropout(dr / 2)(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)
    return Model(inputs=inp, outputs=out, name='CNN_BiGRU')


# ══════════════════════════════════════════════════════════════════════════════
# 4. CNN + BiLSTM  (no attention)
# ══════════════════════════════════════════════════════════════════════════════
def build_cnn_bilstm(input_shape=INPUT_SHAPE, n_classes=N_CLASSES, dr=DR):
    inp = layers.Input(shape=input_shape, name='input')
    x   = cnn_block(inp, 64,  64,  block_id=1)
    x   = cnn_block(x,  128, 128,  block_id=2)
    x   = layers.Bidirectional(
              layers.LSTM(128, return_sequences=True, dropout=0.2), name='bilstm_1')(x)
    x   = layers.Bidirectional(
              layers.LSTM(64,  return_sequences=False, dropout=0.2), name='bilstm_2')(x)
    x   = layers.Dropout(dr)(x)
    x   = layers.Dense(128, name='fc1')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation('relu')(x)
    x   = layers.Dropout(dr / 2)(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)
    return Model(inputs=inp, outputs=out, name='CNN_BiLSTM')


# ══════════════════════════════════════════════════════════════════════════════
# 5. CNN + BiGRU + Gated Attention  ← FLAGSHIP
# ══════════════════════════════════════════════════════════════════════════════
def build_cnn_bigru_gatedattn(input_shape=INPUT_SHAPE, n_classes=N_CLASSES,
                               gru_units_1=128, gru_units_2=64, attn_units=64,
                               dr=DR):
    inp = layers.Input(shape=input_shape, name='input')

    # ── CNN Blocks ────────────────────────────────────────────────────────────
    x = cnn_block(inp, 64,  64,  block_id=1)   # 128→64 timesteps
    x = cnn_block(x,  128, 128,  block_id=2)   # 64→32  timesteps

    # ── Bidirectional GRU ────────────────────────────────────────────────────
    x = layers.Bidirectional(
            layers.GRU(gru_units_1, return_sequences=True, dropout=0.2),
            name='bigru_1')(x)
    x = layers.Bidirectional(
            layers.GRU(gru_units_2, return_sequences=True, dropout=0.2),
            name='bigru_2')(x)

    # ── Gated Attention ───────────────────────────────────────────────────────
    context, _ = GatedAttention(units=attn_units, name='gated_attention')(x)

    # ── Classifier Head ───────────────────────────────────────────────────────
    x   = layers.Dropout(dr)(context)
    x   = layers.Dense(128, name='fc1')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation('relu')(x)
    x   = layers.Dropout(dr / 2)(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)

    return Model(inputs=inp, outputs=out, name='CNN_BiGRU_GatedAttention')


# Print parameter counts
for builder in [build_cnn_only, build_bigru_only, build_cnn_bigru,
                build_cnn_bilstm, build_cnn_bigru_gatedattn]:
    m = builder()
    print(f'{m.name:<35}  params: {m.count_params():>10,}')


## ⚙️ Step 10 — Training Configuration & Utility

In [ ]:
EPOCHS     = 80
BATCH_SIZE = 64
LR_INIT    = 1e-3


def make_lr_schedule():
    return keras.optimizers.schedules.CosineDecayRestarts(
        initial_learning_rate=LR_INIT,
        first_decay_steps=20,
        t_mul=2.0,
        m_mul=0.85,
    )


def make_callbacks(model_name):
    ckpt_path = f'best_{model_name}.weights.h5'
    return [
        callbacks.EarlyStopping(
            monitor='val_accuracy', patience=15,
            restore_best_weights=True, verbose=0,
        ),
        callbacks.ModelCheckpoint(
            ckpt_path, monitor='val_accuracy',
            save_best_only=True, save_weights_only=True, verbose=0,
        ),
    ], ckpt_path


def compile_and_train(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=make_lr_schedule()),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    cb_list, ckpt = make_callbacks(model.name)
    print(f'\n{'='*55}')
    print(f' Training: {model.name}  ({model.count_params():,} params)')
    print(f'{'='*55}')
    history = model.fit(
        X_train, y_train_cat,
        validation_data=(X_val, y_val_cat),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=cb_list,
        verbose=1,
    )
    best_val = max(history.history['val_accuracy'])
    print(f'✅ Done — best val accuracy: {best_val:.4f}')
    return history


def evaluate_model(model, history=None):
    """Return a dict of metrics for all three splits."""
    def _preds(X):
        return model.predict(X, batch_size=BATCH_SIZE, verbose=0)

    proba_test  = _preds(X_test)
    proba_val   = _preds(X_val)
    proba_train = _preds(X_train)

    pred_test   = np.argmax(proba_test,  axis=1)
    pred_val    = np.argmax(proba_val,   axis=1)
    pred_train  = np.argmax(proba_train, axis=1)

    # AUC
    y_test_bin = label_binarize(y_test, classes=list(range(N_CLASSES)))
    auc_scores = []
    for i in range(N_CLASSES):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba_test[:, i])
        auc_scores.append(auc(fpr, tpr))

    metrics = {
        'name'        : model.name,
        'params'      : model.count_params(),
        'train_acc'   : accuracy_score(y_train, pred_train),
        'val_acc'     : accuracy_score(y_val,   pred_val),
        'test_acc'    : accuracy_score(y_test,  pred_test),
        'weighted_f1' : f1_score(y_test, pred_test, average='weighted'),
        'macro_f1'    : f1_score(y_test, pred_test, average='macro'),
        'micro_f1'    : f1_score(y_test, pred_test, average='micro'),
        'kappa'       : cohen_kappa_score(y_test, pred_test),
        'mcc'         : matthews_corrcoef(y_test, pred_test),
        'mean_auc'    : float(np.mean(auc_scores)),
        'auc_scores'  : auc_scores,
        'proba_test'  : proba_test,
        'pred_test'   : pred_test,
        'pred_val'    : pred_val,
        'history'     : history,
    }
    return metrics


print('Training utilities ready.')
print(f'  Epochs     : {EPOCHS}')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  LR init    : {LR_INIT}')


## 🚀 Step 11 — Train Model 1 — CNN Only

In [ ]:
model_cnn = build_cnn_only()
model_cnn.summary()
hist_model_cnn = compile_and_train(model_cnn)
metrics_model_cnn  = evaluate_model(model_cnn, hist_model_cnn)

print(f'\n  Train Acc : {metrics_model_cnn["train_acc"]:.4f}')
print(f'  Val   Acc : {metrics_model_cnn["val_acc"]:.4f}')
print(f'  Test  Acc : {metrics_model_cnn["test_acc"]:.4f}')
print(f'  Weighted F1: {metrics_model_cnn["weighted_f1"]:.4f}')
print(f'  Mean AUC  : {metrics_model_cnn["mean_auc"]:.4f}')


## 🚀 Step 12 — Train Model 2 — BiGRU Only

In [ ]:
model_bigru = build_bigru_only()
model_bigru.summary()
hist_model_bigru = compile_and_train(model_bigru)
metrics_model_bigru  = evaluate_model(model_bigru, hist_model_bigru)

print(f'\n  Train Acc : {metrics_model_bigru["train_acc"]:.4f}')
print(f'  Val   Acc : {metrics_model_bigru["val_acc"]:.4f}')
print(f'  Test  Acc : {metrics_model_bigru["test_acc"]:.4f}')
print(f'  Weighted F1: {metrics_model_bigru["weighted_f1"]:.4f}')
print(f'  Mean AUC  : {metrics_model_bigru["mean_auc"]:.4f}')


## 🚀 Step 13 — Train Model 3 — CNN + BiGRU

In [ ]:
model_cnn_bigru = build_cnn_bigru()
model_cnn_bigru.summary()
hist_model_cnn_bigru = compile_and_train(model_cnn_bigru)
metrics_model_cnn_bigru  = evaluate_model(model_cnn_bigru, hist_model_cnn_bigru)

print(f'\n  Train Acc : {metrics_model_cnn_bigru["train_acc"]:.4f}')
print(f'  Val   Acc : {metrics_model_cnn_bigru["val_acc"]:.4f}')
print(f'  Test  Acc : {metrics_model_cnn_bigru["test_acc"]:.4f}')
print(f'  Weighted F1: {metrics_model_cnn_bigru["weighted_f1"]:.4f}')
print(f'  Mean AUC  : {metrics_model_cnn_bigru["mean_auc"]:.4f}')


## 🚀 Step 14 — Train Model 4 — CNN + BiLSTM

In [ ]:
model_cnn_bilstm = build_cnn_bilstm()
model_cnn_bilstm.summary()
hist_model_cnn_bilstm = compile_and_train(model_cnn_bilstm)
metrics_model_cnn_bilstm  = evaluate_model(model_cnn_bilstm, hist_model_cnn_bilstm)

print(f'\n  Train Acc : {metrics_model_cnn_bilstm["train_acc"]:.4f}')
print(f'  Val   Acc : {metrics_model_cnn_bilstm["val_acc"]:.4f}')
print(f'  Test  Acc : {metrics_model_cnn_bilstm["test_acc"]:.4f}')
print(f'  Weighted F1: {metrics_model_cnn_bilstm["weighted_f1"]:.4f}')
print(f'  Mean AUC  : {metrics_model_cnn_bilstm["mean_auc"]:.4f}')


## 🚀 Step 15 — Train Model 5 — CNN + BiGRU + Gated Attention ⭐

In [ ]:
model_flagship = build_cnn_bigru_gatedattn()
model_flagship.summary()
hist_model_flagship = compile_and_train(model_flagship)
metrics_model_flagship  = evaluate_model(model_flagship, hist_model_flagship)

print(f'\n  Train Acc : {metrics_model_flagship["train_acc"]:.4f}')
print(f'  Val   Acc : {metrics_model_flagship["val_acc"]:.4f}')
print(f'  Test  Acc : {metrics_model_flagship["test_acc"]:.4f}')
print(f'  Weighted F1: {metrics_model_flagship["weighted_f1"]:.4f}')
print(f'  Mean AUC  : {metrics_model_flagship["mean_auc"]:.4f}')


## 📊 Step 16 — Collect All Results

In [ ]:
all_results = [metrics_model_cnn, metrics_model_bigru, metrics_model_cnn_bigru, metrics_model_cnn_bilstm, metrics_model_flagship]
print('All models evaluated.')
for r in all_results:
    print(f"  {r['name']:<40} test_acc={r['test_acc']:.4f}  wF1={r['weighted_f1']:.4f}")


## 📈 Step 17 — Training Curves (All Models)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(26, 8))

for col, res in enumerate(all_results):
    hist    = res['history'].history
    epochs_ = range(1, len(hist['accuracy']) + 1)
    best_ep = np.argmax(hist['val_accuracy']) + 1

    ax_acc = axes[0, col]
    ax_los = axes[1, col]

    ax_acc.plot(epochs_, hist['accuracy'],     color='royalblue', lw=1.8, label='Train')
    ax_acc.plot(epochs_, hist['val_accuracy'], color='tomato',    lw=1.8, label='Val')
    ax_acc.axvline(best_ep, color='green', ls='--', alpha=0.7)
    ax_acc.set_title(res['name'].replace('_', '\n'), fontsize=9, fontweight='bold')
    ax_acc.set_ylabel('Accuracy') if col == 0 else None
    ax_acc.legend(fontsize=7); ax_acc.grid(alpha=0.3)
    ax_acc.text(0.98, 0.02, f"best={max(hist['val_accuracy']):.3f}",
                transform=ax_acc.transAxes, ha='right', va='bottom', fontsize=8, color='green')

    ax_los.plot(epochs_, hist['loss'],     color='royalblue', lw=1.8, label='Train')
    ax_los.plot(epochs_, hist['val_loss'], color='tomato',    lw=1.8, label='Val')
    ax_los.axvline(best_ep, color='green', ls='--', alpha=0.7)
    ax_los.set_xlabel('Epoch'); ax_los.grid(alpha=0.3)
    ax_los.set_ylabel('Loss') if col == 0 else None

fig.suptitle('Training History — All 5 Models (MHEALTH)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('mhealth_training_curves_all.png', dpi=130, bbox_inches='tight')
plt.show()


## 🏆 Step 18 — Model Comparison Table

In [ ]:
print('\n' + '='*100)
print(f'  {"Model":<40} {"Params":>10} {"Train Acc":>10} {"Val Acc":>9} {"Test Acc":>9} {"wF1":>7} {"AUC":>7} {"Kappa":>7} {"MCC":>7}')
print('-'*100)
for r in all_results:
    gap_flag = ' ⚠️' if (r['train_acc'] - r['test_acc']) > 0.07 else ''
    print(f"  {r['name']:<40} {r['params']:>10,} {r['train_acc']:>10.4f} {r['val_acc']:>9.4f} "          f"{r['test_acc']:>9.4f} {r['weighted_f1']:>7.4f} {r['mean_auc']:>7.4f} "          f"{r['kappa']:>7.4f} {r['mcc']:>7.4f}{gap_flag}")
print('='*100)

best = max(all_results, key=lambda r: r['test_acc'])
print(f"\n🏆 Best model: {best['name']}  →  Test Acc={best['test_acc']:.4f}  wF1={best['weighted_f1']:.4f}")


## 📊 Step 19 — Visual Comparison Bar Chart

In [ ]:
metrics_to_plot = ['test_acc', 'weighted_f1', 'macro_f1', 'mean_auc', 'kappa', 'mcc']
metric_labels   = ['Test Accuracy', 'Weighted F1', 'Macro F1', 'Mean AUC', "Cohen's Kappa", 'MCC']

model_names_short = ['CNN\nOnly', 'BiGRU\nOnly', 'CNN+\nBiGRU', 'CNN+\nBiLSTM', 'CNN+BiGRU\n+GatedAttn']
x = np.arange(len(all_results))
width = 0.13
colors_bar = ['steelblue', 'mediumseagreen', 'darkorange', 'orchid', 'tomato', 'goldenrod']

fig, ax = plt.subplots(figsize=(16, 6))
for j, (m_key, m_label, c) in enumerate(zip(metrics_to_plot, metric_labels, colors_bar)):
    vals = [r[m_key] for r in all_results]
    offset = (j - len(metrics_to_plot)/2 + 0.5) * width
    bars = ax.bar(x + offset, vals, width, label=m_label, color=c, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{v:.3f}', ha='center', va='bottom', fontsize=6, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(model_names_short, fontsize=10)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('MHEALTH — Model Comparison: All Metrics', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9, ncol=2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('mhealth_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 🟦 Step 20 — Confusion Matrices (All Models)

In [ ]:
def plot_confusion_matrix(y_true, y_pred_labels, model_name, save_path):
    cm   = confusion_matrix(y_true, y_pred_labels)
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    labels = [ACTIVITY_LABELS[i+1] for i in range(N_CLASSES)]

    fig, axes = plt.subplots(1, 2, figsize=(22, 7))
    for ax, data, fmt, ttl in zip(axes,
                                   [cm, cm_n],
                                   ['d', '.1%'],
                                   ['Raw Counts', 'Normalized (Row %)']):
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=labels, yticklabels=labels,
                    linewidths=0.4, ax=ax, annot_kws={'size': 7})
        ax.set_title(f'{model_name} — {ttl}', fontsize=11, fontweight='bold')
        ax.set_ylabel('True Label', fontsize=9)
        ax.set_xlabel('Predicted Label', fontsize=9)
        ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=7)
        ax.set_yticklabels(labels, rotation=0, fontsize=7)

    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()


for res in all_results:
    plot_confusion_matrix(
        y_test, res['pred_test'],
        res['name'],
        f"mhealth_cm_{res['name'].replace(' ', '_').replace('+', '')}.png"
    )


## 📉 Step 21 — ROC Curves (One-vs-Rest, All Models)

In [ ]:
y_test_bin = label_binarize(y_test, classes=list(range(N_CLASSES)))

fig, axes = plt.subplots(1, 5, figsize=(28, 5))

for ax, res in zip(axes, all_results):
    tab_colors = plt.cm.tab20(np.linspace(0, 1, N_CLASSES))
    for i, (act_name, c) in enumerate(zip(ACTIVITY_LABELS.values(), tab_colors)):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], res['proba_test'][:, i])
        ax.plot(fpr, tpr, color=c, lw=1.5, label=f'{act_name[:10]} ({res["auc_scores"][i]:.2f})')
    ax.plot([0,1],[0,1],'k--', lw=1)
    ax.set_title(f'{res["name"]}\nmAUC={res["mean_auc"]:.4f}', fontsize=9, fontweight='bold')
    ax.set_xlabel('FPR', fontsize=8); ax.set_ylabel('TPR', fontsize=8)
    ax.legend(fontsize=5, loc='lower right'); ax.grid(alpha=0.3)

fig.suptitle('ROC Curves — Test Set (All Models, MHEALTH)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('mhealth_roc_all_models.png', dpi=130, bbox_inches='tight')
plt.show()


## 📊 Step 22 — Per-Class Precision / Recall / F1 (All Models)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(30, 6))
x_cls = np.arange(N_CLASSES)
width = 0.28

for ax, res in zip(axes, all_results):
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_test, res['pred_test'], average=None, labels=list(range(N_CLASSES)))
    ax.bar(x_cls - width, prec, width, label='Precision', color='steelblue',      alpha=0.85)
    ax.bar(x_cls,         rec,  width, label='Recall',    color='mediumseagreen', alpha=0.85)
    ax.bar(x_cls + width, f1,   width, label='F1',        color='tomato',         alpha=0.85)
    ax.set_title(res['name'], fontsize=8, fontweight='bold')
    ax.set_xticks(x_cls)
    ax.set_xticklabels([ACTIVITY_LABELS[i+1][:6] for i in range(N_CLASSES)],
                        rotation=45, ha='right', fontsize=6)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel('Score')
    ax.legend(fontsize=6)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Per-Class Precision / Recall / F1 — Test Set (MHEALTH)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('mhealth_perclass_all.png', dpi=120, bbox_inches='tight')
plt.show()


## 🔢 Step 23 — Classification Reports (All Models)

In [ ]:
for res in all_results:
    print('\n' + '='*70)
    print(f' Classification Report — {res["name"]}')
    print('='*70)
    print(classification_report(
        y_test, res['pred_test'],
        target_names=[ACTIVITY_LABELS[i+1] for i in range(N_CLASSES)],
        digits=4,
    ))


## 👁️ Step 24 — Gated Attention Weights Visualisation (Flagship Model)

In [ ]:
# Build a sub-model that exposes the attention weights from the flagship model
gru_out    = model_flagship.get_layer('bigru_2').output
attn_layer = model_flagship.get_layer('gated_attention')
_, attn_out = attn_layer(gru_out)

attn_model = Model(inputs=model_flagship.input, outputs=attn_out)

fig, axes = plt.subplots(3, 4, figsize=(22, 12))

for ax, act_idx in zip(axes.flatten(), range(N_CLASSES)):
    # Pick first test sample of this activity
    candidates = np.where(y_test == act_idx)[0]
    if len(candidates) == 0:
        ax.set_title(f'{ACTIVITY_LABELS[act_idx+1]} (no sample)', fontsize=8)
        ax.axis('off'); continue

    idx    = candidates[0]
    sample = X_test[idx:idx+1]                            # (1, 128, 23)
    attn_w = attn_model.predict(sample, verbose=0)[0]     # (T_reduced,)
    T_red  = len(attn_w)
    x_ticks = np.linspace(0, WINDOW_SIZE, T_red)

    raw_sig = sample[0, :, 0]                             # chest_acc_x
    ax2 = ax.twinx()
    ax.plot(raw_sig, color='steelblue', lw=1.2, label='chest_acc_x')
    ax2.bar(x_ticks, attn_w, width=3.5, color='tomato', alpha=0.55, label='Attention')
    ax2.set_ylabel('Attn weight', color='tomato', fontsize=7)
    ax.set_title(f'{ACTIVITY_LABELS[act_idx+1]}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Timestep', fontsize=7)
    ax.grid(alpha=0.2)

fig.suptitle(
    'Gated Attention Weights per Activity (Flagship Model)\n'
    'Red bars = attention weight | Blue line = chest_acc_x',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('mhealth_attention_viz.png', dpi=130, bbox_inches='tight')
plt.show()


## 🏆 Step 25 — Final Summary Dashboard

In [ ]:
print()
print('█'*80)
print('   FINAL RESULTS — MHEALTH HUMAN ACTIVITY RECOGNITION')
print('   5-Model Comparison: CNN · BiGRU · CNN+BiGRU · CNN+BiLSTM · CNN+BiGRU+GatedAttn')
print('█'*80)
print(f'  Dataset          : MHEALTH')
print(f'  Subjects         : {N_SUBJECTS} (7 train · 1 val · 2 test — subject-level, no leakage)')
print(f'  Activities       : {N_CLASSES} (null class excluded)')
print(f'  Sensor channels  : {N_CHANNELS}')
print(f'  Window size      : {WINDOW_SIZE} samples @ {SAMPLING_HZ} Hz = {WINDOW_SIZE/SAMPLING_HZ:.2f} s')
print(f'  Step size        : {STEP_SIZE} samples (50% overlap)')
print(f'  Train windows    : {len(X_train):,}')
print(f'  Val   windows    : {len(X_val):,}')
print(f'  Test  windows    : {len(X_test):,}')
print()
print(f'  {"Model":<40} {"Test Acc":>10} {"wF1":>8} {"mAUC":>8} {"Kappa":>8} {"MCC":>8} {"Params":>12}')
print('  ' + '-'*98)
for r in all_results:
    gap = r['train_acc'] - r['test_acc']
    flag = ' ✅' if gap < 0.05 else ' ⚠️' if gap < 0.10 else ' ❌'
    print(f"  {r['name']:<40} {r['test_acc']:>10.4f} {r['weighted_f1']:>8.4f} {r['mean_auc']:>8.4f} "          f"{r['kappa']:>8.4f} {r['mcc']:>8.4f} {r['params']:>12,}{flag}")

print()
best = max(all_results, key=lambda r: r['test_acc'])
print(f'  🏆 BEST MODEL : {best["name"]}')
print(f'     Test Accuracy  : {best["test_acc"]:.4f}')
print(f'     Weighted F1    : {best["weighted_f1"]:.4f}')
print(f'     Mean AUC (OvR) : {best["mean_auc"]:.4f}')
print(f"     Train-Test gap : {best['train_acc'] - best['test_acc']:.4f}")
print('█'*80)


## 💾 Step 26 — Save Results & Download Bundle

In [ ]:
from google.colab import files

# Save all models
for i, (m, var_name) in enumerate(zip(
    [model_cnn, model_bigru, model_cnn_bigru, model_cnn_bilstm, model_flagship],
    ['cnn_only', 'bigru_only', 'cnn_bigru', 'cnn_bilstm', 'cnn_bigru_gatedattn']
)):
    m.save(f'mhealth_{var_name}.keras')

# Save results JSON
results_json = []
for r in all_results:
    results_json.append({
        'model'        : r['name'],
        'params'       : r['params'],
        'train_acc'    : float(r['train_acc']),
        'val_acc'      : float(r['val_acc']),
        'test_acc'     : float(r['test_acc']),
        'weighted_f1'  : float(r['weighted_f1']),
        'macro_f1'     : float(r['macro_f1']),
        'micro_f1'     : float(r['micro_f1']),
        'mean_auc'     : float(r['mean_auc']),
        'kappa'        : float(r['kappa']),
        'mcc'          : float(r['mcc']),
        'per_class_auc': {ACTIVITY_LABELS[i+1]: float(s)
                          for i, s in enumerate(r['auc_scores'])},
    })

with open('mhealth_all_results.json', 'w') as f:
    json.dump(results_json, f, indent=2)
print('mhealth_all_results.json saved.')

# Bundle into ZIP
output_files = [
    'mhealth_all_results.json',
    'mhealth_training_curves_all.png',
    'mhealth_model_comparison.png',
    'mhealth_roc_all_models.png',
    'mhealth_perclass_all.png',
    'mhealth_attention_viz.png',
    'mhealth_class_distribution.png',
    'mhealth_signal_sample.png',
] + [f'mhealth_{n}.keras' for n in ['cnn_only','bigru_only','cnn_bigru','cnn_bilstm','cnn_bigru_gatedattn']
] + [f'mhealth_cm_{r["name"].replace(" ","_").replace("+","")}.png' for r in all_results]

with zipfile.ZipFile('MHEALTH_results.zip', 'w') as zf:
    for fname in output_files:
        if os.path.exists(fname):
            zf.write(fname)

print('All outputs bundled → MHEALTH_results.zip')
files.download('MHEALTH_results.zip')


---
## 📌 Architecture & Methodology Notes

### Dataset
| Property | Value |
|---|---|
| Source | MHEALTH (Mobile HEALTH) dataset |
| Subjects | 10 |
| Activities | 12 (null class excluded) |
| Sensors | Chest acc · Left-ankle (acc+gyro+mag) · Right-arm (acc+gyro+mag) · ECG |
| Channels | 23 |
| Sampling rate | 50 Hz |
| Window size | 128 samples = 2.56 s |
| Overlap | 50 % (step = 64) |
| Window label | Majority vote of sample labels |

### Models
| Model | Architecture |
|---|---|
| **CNN Only** | 3× (Conv1D×2 + BN + ReLU + MaxPool + Dropout) → GAP → Dense |
| **BiGRU Only** | BiGRU(128, seq) → BiGRU(64, no-seq) → Dense |
| **CNN + BiGRU** | 2× CNN block → BiGRU(128, seq) → BiGRU(64, no-seq) → Dense |
| **CNN + BiLSTM** | 2× CNN block → BiLSTM(128, seq) → BiLSTM(64, no-seq) → Dense |
| **CNN + BiGRU + Gated Attention** ⭐ | 2× CNN block → BiGRU(128, seq) → BiGRU(64, seq) → GatedAttention(64) → Dense |

### Gated Attention
| Component | Formula |
|---|---|
| Content gate | score = tanh(H @ W_h) |
| Information gate | gate = σ(H @ W_g) |
| Attention weights | α = softmax(score ⊙ gate) |
| Context vector | c = Σ_t(α_t × h_t) |

### Training Protocol
| Setting | Value |
|---|---|
| Optimizer | Adam + Cosine Decay Restarts |
| Initial LR | 1e-3 |
| Batch size | 64 |
| Max epochs | 80 |
| Early stopping | patience=15, monitor val_accuracy |
| Regularization | Dropout (0.4), Batch Normalization |
| Normalization | Per-channel Z-score (fit on train only) |
| Split strategy | Subject-level — no subject overlap |
